In [ ]:
from py2neo import Node, Relationship, Graph, NodeMatcher, RelationshipMatcher, Subgraph
g = Graph('bolt://localhost:7687',auth=('neo4j','your password'), name = 'trkg')

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

# ---------- locate project root ----------
def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if (parent / "Main data").exists():
            return parent
    raise RuntimeError("Project root (CODE UPLOAD) not found.")

PROJECT_ROOT = find_project_root()

# ---------- relative CSV path ----------
csv_path = (
    PROJECT_ROOT
    / "shanghai_data_save"
    / "local_grid_time-series"
    / "completed_tensor_area.csv"
)

df = pd.read_csv(csv_path)

column_names = df.columns.tolist()[1:]
column_names_cypher = ", ".join(str(name) for name in column_names)
column_names_cypher

'92973, 92974, 92986, 92989, 92992, 92994, 92995, 92996, 93001, 93002, 93003, 93016, 93017, 93022, 93592, 93597, 93600, 93601, 93621, 93625, 93627, 93629, 93630, 93631, 93632, 93653, 93665, 93666, 93667, 93669, 93671, 93676, 93682, 93684, 93697, 93699, 93700, 93702, 93703, 93707, 93708, 93709, 93711, 93712, 93713, 93714, 93716, 93719, 93723, 93724, 93725, 93727, 93728, 93729, 93730, 93734, 93839, 93840, 93842, 93844, 93845, 93848, 93852, 93855, 93859, 93862, 93863, 93870, 93871, 93872, 93873, 93876, 93885, 93891, 93893, 93897, 93898, 93899, 93901, 93904, 93908, 93909, 93910, 93911, 93912, 93915, 93916, 93918, 93919, 93922, 93924, 93925, 93927, 93932, 93933, 93935, 93936, 93937, 93938, 93939, 93940, 93942, 93943, 93944, 93945, 93948, 93954, 93955, 93958, 93960, 93961, 93962, 93965, 93967, 93969, 93971, 93972, 93974, 93975, 93977, 93979, 93980, 93981, 93983, 93984, 93987, 93988, 93989, 93992, 93993, 93994, 93996, 93998, 93999, 94001, 94002, 94005, 94008, 94009, 94011, 94012, 94013, 94015

In [11]:
import numpy as np

# Step 1: 查询原始连接边对
query = f"""
MATCH (h1:honeycomb)<-[:within]-(r1:road)-[:touch]-(r2:road)-[:within]->(h2:honeycomb)
WHERE h1.name < h2.name
  AND h1.name IN [{column_names_cypher}]
  AND h2.name IN [{column_names_cypher}]
RETURN DISTINCT h1.name AS h1_name, h2.name AS h2_name
"""
results = g.run(query).data()

# Step 2: 提取所有已连接的 honeycomb 名
all_ids = set()
for row in results:
    all_ids.add(row['h1_name'])
    all_ids.add(row['h2_name'])

# Step 3: 获取所有需要的 honeycomb 名（来自 column_names_cypher）
column_names_set = set(int(x.strip("'")) for x in column_names_cypher.split(','))

# Step 4: 识别未连接的 honeycomb
unconnected_ids = sorted(column_names_set - all_ids)
print(f"⚠️ 未连接 honeycomb 数量: {len(unconnected_ids)}")
print(f"⚠️ 未连接 honeycomb : {unconnected_ids}")

# Step 5: 构建连接 ID 列表和索引映射（后续使用）
id_list = sorted(all_ids)
id_to_idx = {hid: idx for idx, hid in enumerate(id_list)}

# Step 6: 补建未连接的 honeycomb
recovered_links = []

for hid in unconnected_ids:
    query = f"""
    MATCH (h:honeycomb {{name: {hid}}}),
          (h2:honeycomb)
    WHERE h2.name IN [{", ".join(map(str, id_list))}]
    MATCH p = shortestPath((h)-[:adjacency*1..10]-(h2))
    RETURN h.name AS h1_name, h2.name AS h2_name
    LIMIT 1
    """
    try:
        data = g.run(query).data()
        if data:
            h1 = data[0]['h1_name']
            h2 = data[0]['h2_name']
            recovered_links.append((h1, h2))
            all_ids.add(h1)
            all_ids.add(h2)
            print(f"✅ 补建成功：{h1} <-> {h2}")
        else:
            print(f"⚠️ 无路径可补建: {hid}")
    except Exception as e:
        print(f"❌ shortestPath 查询失败 {hid}: {e}")

print(f"✅ 成功补建的 honeycomb 数量: {len(recovered_links)}")

# Step 7: 更新所有 ID 映射（包括补建后的）
id_list = sorted(all_ids)
id_to_idx = {hid: idx for idx, hid in enumerate(id_list)}
n = len(id_list)
print(f"总节点数（含补建）: {n}")

# Step 8: 初始化邻接矩阵
adj_dense = np.zeros((n, n), dtype=np.float32)

# Step 9: 添加原始连接边
for row in results:
    i = id_to_idx[row['h1_name']]
    j = id_to_idx[row['h2_name']]
    adj_dense[i, j] = 1
    adj_dense[j, i] = 1

# Step 10: 添加补建边
for h1, h2 in recovered_links:
    i = id_to_idx[h1]
    j = id_to_idx[h2]
    adj_dense[i, j] = 1
    adj_dense[j, i] = 1

# Step 11: 输出邻接矩阵结果
print("✅ 构建完成的邻接矩阵：")
print(adj_dense)
print("矩阵维度：", adj_dense.shape)


⚠️ 未连接 honeycomb 数量: 5
⚠️ 未连接 honeycomb : [93665, 95197, 95307, 95309, 95476]
✅ 补建成功：93665 <-> 93842
✅ 补建成功：95197 <-> 95207
✅ 补建成功：95307 <-> 95207
✅ 补建成功：95309 <-> 95207
✅ 补建成功：95476 <-> 95207
✅ 成功补建的 honeycomb 数量: 5
总节点数（含补建）: 265
✅ 构建完成的邻接矩阵：
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
矩阵维度： (265, 265)


In [12]:
# 构建邻接矩阵 DataFrame
df_adj = pd.DataFrame(adj_dense, index=id_list, columns=id_list)

# ---------- OUTPUT 1: 带行列名的 CSV（relative） ----------
out_dir_1 = PROJECT_ROOT / "shanghai_data_save" / "local_grid_time-series"
out_dir_1.mkdir(parents=True, exist_ok=True)

out_csv_1 = out_dir_1 / "adj_matrix_dense.csv"
df_adj.to_csv(out_csv_1, index=True, encoding="utf-8-sig")
print("✅ 已保存为 .csv 格式")

# ---------- OUTPUT 2: 纯数值 CSV（relative，用于模型输入） ----------
out_dir_2 = PROJECT_ROOT / "Main data" / "prediction_data"
out_dir_2.mkdir(parents=True, exist_ok=True)

out_csv_2 = out_dir_2 / "adj_matrix.csv"
df_adj.to_csv(out_csv_2, index=False, header=False, encoding="utf-8-sig")
print("✅ 已保存为纯数值格式 adj_matrix.csv（prediction_data）")

✅ 已保存为 .csv 格式
✅ 已保存为纯数值格式 adj_matrix.csv


In [13]:
csv_path = (
    PROJECT_ROOT
    / "shanghai_data_save"
    / "local_grid_time-series"
    / "adj_matrix_dense.csv"
)

# 读取 CSV 格式的邻接矩阵
df_adj = pd.read_csv(csv_path, index_col=0)

# 转为 NumPy 数组（邻接矩阵）
adj_dense = df_adj.values

df_adj

,92973,92974,92986,92989,92992,92994,92995,92996,93001,93002,...,95445,95448,95449,95450,95451,95453,95465,95468,95475,95476
92973,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
92974,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
92986,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
92989,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
92992,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95453,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
95465,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
95468,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
95475,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
